# SVD (Matrix Factorization via Surprise)

Funk-SVD-style matrix factorization (Koren et al., 2009) as implemented in the [Surprise](https://surpriselib.com/) library. We learn user latent factors `p_u ∈ R^k`, item latent factors `q_i ∈ R^k`, user biases `b_u`, item biases `b_i`, and global mean `μ`, with the prediction

$$ \hat{r}_{ui} = \mu + b_u + b_i + p_u^\top q_i, $$

minimizing

$$ \sum_{(u,i)\in\mathcal{T}} \big(r_{ui} - \hat{r}_{ui}\big)^2 + \lambda\big(\|p_u\|^2 + \|q_i\|^2 + b_u^2 + b_i^2\big) $$

via stochastic gradient descent.


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
!pip install -q "numpy<2" scikit-surprise==1.1.4



In [3]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

## Load splits

In [4]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}')

train: 99,616  val: 610  test: 610


In [5]:
# Surprise wants (user, item, rating) triples; rating scale must be declared.
reader = Reader(rating_scale=(0.5, 5.0))

trainval = pd.concat([train, val], ignore_index=True)
data_tv = Dataset.load_from_df(trainval[['userId', 'movieId', 'rating']], reader)
data_tr = Dataset.load_from_df(train[['userId', 'movieId', 'rating']],    reader)

## Hyperparameter sweep (3-fold CV on train+val)

Searches over latent-factor count, learning rate, regularization, and number of epochs. 3 folds rather than 5 because `ml-latest-small` has only 610 users; with 5 folds the per-fold variance dominates the parameter signal.

In [6]:
param_grid = {
    'n_factors':  [50, 100, 150],
    'n_epochs':   [20, 30],
    'lr_all':     [0.005, 0.01],
    'reg_all':    [0.02, 0.06],
    'random_state': [SEED],
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1, joblib_verbose=1)
gs.fit(data_tv)

best_params = gs.best_params['rmse']
print('Best RMSE (CV):', round(gs.best_score['rmse'], 4))
print('Best MAE  (CV):', round(gs.best_score['mae'],  4))
print('Best params   :', best_params)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.1min


Best RMSE (CV): 0.8627
Best MAE  (CV): 0.6616
Best params   : {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.06, 'random_state': 42}


[Parallel(n_jobs=-1)]: Done  72 out of  72 | elapsed:  1.8min finished


In [7]:
# Inspect the top-5 hyperparameter configurations.
results_df = pd.DataFrame.from_dict(gs.cv_results)
cols = ['mean_test_rmse', 'mean_test_mae'] + [c for c in results_df.columns if c.startswith('param_')]
results_df[cols].sort_values('mean_test_rmse').head(5)

,mean_test_rmse,mean_test_mae,param_n_factors,param_n_epochs,param_lr_all,param_reg_all,param_random_state
23,0.862708,0.661604,150,30,0.01,0.06,42
15,0.863597,0.661965,100,30,0.01,0.06,42
7,0.865883,0.663780,50,30,0.01,0.06,42
19,0.865915,0.664802,150,20,0.01,0.06,42
11,0.866459,0.664952,100,20,0.01,0.06,42


## Final fit on train+val with best hyperparameters

In [8]:
model = SVD(**best_params)
model.fit(data_tv.build_full_trainset())
print(f'Trained SVD with: {best_params}')

Trained SVD with: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.06, 'random_state': 42}


## Predict on test set + run full metric suite

In [9]:
def predict_pointwise(df):
    """Per-row predicted rating for the (userId, movieId) pairs in df."""
    out = df[['userId', 'movieId', 'rating']].copy()
    out['predicted_rating'] = [model.predict(uid, iid).est for uid, iid in zip(df['userId'], df['movieId'])]
    return out

def predict_fn(user_id, movie_ids):
    """Score for ranking metrics. Surprise's predict() handles unseen users/items gracefully."""
    return np.array([model.predict(int(user_id), int(m)).est for m in movie_ids], dtype=float)

In [10]:
# Candidate lists: train is the 'seen' set; sampled negatives drawn from items neither in train nor the test interaction.
candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)
print(f'Built candidate lists for {len(candidates)} test interactions.')

Built candidate lists for 610 test interactions.


In [11]:
pointwise = predict_pointwise(test)
metrics = evaluate_model(predict_fn, test, candidates, pointwise_predictions=pointwise, k=10, threshold=3.5)

print('=== SVD — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics[key]:.4f}')

=== SVD — test set ===
  rmse               = 0.9412
  mae                = 0.7278
  hr_at_k            = 0.2705
  ndcg_at_k          = 0.1515
  precision_at_k     = 0.0226
  recall_at_k        = 0.2262
  f1_at_k            = 0.0411


## Save predictions

In [12]:
out_path = PREDICTIONS_DIR / 'svd.csv'
pointwise[['userId', 'movieId', 'predicted_rating']].to_csv(out_path, index=False)
print(f'Saved {len(pointwise)} predictions -> {out_path}')

Saved 610 predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/svd.csv
